# Chapter 02: Matrices and Linear Systems

Source orientation: printed pages 9-62; PDF pages 56-109.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. The goal is not to reproduce the book; the goal is to turn the chapter into an inspectable graphics-geometry lab. A reader should be able to learn the main ideas here with no PDF open.

In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Tools-for-Computer-Graphics/part-01-foundations/chapter-02-matrices-and-linear-systems/02-matrices-and-linear-systems.ipynb",
  "course_dir": "Geometric-Tools-for-Computer-Graphics",
  "course_title": "Geometric Tools for Computer Graphics",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Tools-for-Computer-Graphics/part-01-foundations/chapter-02-matrices-and-linear-systems/02-matrices-and-linear-systems.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Geometric-Tools-for-Computer-Graphics/part-01-foundations/chapter-02-matrices-and-linear-systems/02-matrices-and-linear-systems.ipynb",
  "notebook_title": "Chapter 02: Matrices and Linear Systems",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/graphics.txt",
  "runtime_profile": "graphics"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Chapter Goal

Linear maps, row reduction, rank, determinants, eigenspaces, Euclidean inner products, and least-squares fitting.

Geometric tools for graphics are easiest to remember when each formula is connected to a representation, a picture, and a check. This lesson therefore treats the chapter as a sequence of decisions a programmer makes: how to represent the primitive, which parameters define the query, what degeneracies are possible, and how to verify that the computed answer is still geometric. The same discipline applies to renderers, modeling tools, collision systems, curve editors, CAD importers, and mesh-processing code. Throughout the notebook, formulas are rewritten as small executable experiments so that the main concept can be rotated, sampled, plotted, and tested.

The linear algebra chapter is organized around what matrices do to spaces. A grid deformation shows the map, a determinant check measures oriented area, and a least-squares discussion interprets residuals as perpendicular leftover vectors. The chapter treats row operations, rank, eigenvectors, and decompositions as ways to reveal structure before a geometric algorithm depends on it.

## Translation Guide

- **Representation:** choose arrays, frames, equations, graphs, or meshes that make Matrices and Linear Systems inspectable.
- **Domain:** identify whether parameters are free, clamped, periodic, barycentric, bounded by a simplex, or constrained by a surface.
- **Invariant:** decide what should not change under translation, rotation, reparameterization, input order, or coordinate-system choice.
- **Residual:** convert the invariant into a number that can be asserted after the figure is drawn.
- **Failure mode:** expose degeneracy, near-zero denominators, endpoint cases, tangencies, or rank loss instead of hiding them behind a single Boolean answer.

This guide is intentionally repeated across the course because it is the working style that makes a reference book become a usable notebook course. Each chapter changes the objects and algorithms, but the learning loop remains the same: model, draw, perturb, measure, and check.

## Route Through The Chapter

1. Translate the chapter vocabulary into computational objects.
2. Build visual artifacts that expose the main invariants.
3. Run a small numeric experiment that makes stability or classification visible.
4. Close with sanity checks that make the notebook reproducible.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the GTCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-02"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

## Visual Storyboard

- grid deformation by a linear map
- row-reduction pivots and rank
- determinant area scale and orientation
- least-squares residual as an orthogonal component

Each planned visual names the geometric behavior to inspect, not just the rendering technology. Static PNG artifacts are used for durable diagrams. If future work adds interactive widgets or Plotly scenes, they should still write stable HTML artifacts under the same chapter artifact subtree.

## Core Concepts

### 1. A matrix is a coordinate representation of a linear map, not the geometry itself

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **grid deformation by a linear map**, so the reader can inspect the construction rather than only read a formula. The paired check is **matrix area scale matches the determinant**, which turns the claim into an executable invariant. In Matrices and Linear Systems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 2. Rank and row reduction expose how many independent constraints a system contains

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **row-reduction pivots and rank**, so the reader can inspect the construction rather than only read a formula. The paired check is **least-squares residual is orthogonal to the column space**, which turns the claim into an executable invariant. In Matrices and Linear Systems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 3. Determinants measure oriented scale and warn when inverse problems become ill conditioned

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **determinant area scale and orientation**, so the reader can inspect the construction rather than only read a formula. The paired check is **row-reduction rank matches numpy matrix rank**, which turns the claim into an executable invariant. In Matrices and Linear Systems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

### 4. Least squares replaces impossible exact constraints with a projection onto a column space

Computational interpretation: this claim becomes a specific data contract in the notebook. The paired visual is **least-squares residual as an orthogonal component**, so the reader can inspect the construction rather than only read a formula. The paired check is **eigenvectors reproduce their scaled directions**, which turns the claim into an executable invariant. In Matrices and Linear Systems, this matters because a geometry program must preserve the distinction between representation, domain, and geometric meaning; otherwise the same arrays can produce a plausible picture and still violate the intended query.

## Worked Example Pattern

The worked example below uses compact synthetic data rather than copied textbook figures. That is deliberate: a synthetic example can be regenerated, perturbed, and checked. The first artifact is a concept map that connects the chapter goal to the planned visuals. The second artifact is a geometric scene specialized to Matrices and Linear Systems. The third artifact is a numeric diagnostic that turns a qualitative claim into a curve or residual. When a chapter contains many algorithms, this pattern becomes a template for further exploration: choose a primitive, construct a small query, draw the active features, then assert the invariants that justify the returned answer. The examples are small enough to read but structured enough that a learner can swap in their own points, matrices, curves, surfaces, or meshes.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from utils.artifacts import display_artifact, save_json, save_matplotlib
from utils.chapter_visuals import compute_check_values, concept_map_figure, geometry_scene_figure, numerical_experiment_figure, storyboard_gallery_figure
from utils.validation import artifact_report, require_nonempty

ENTRY_TITLE = 'Matrices and Linear Systems'
MODE = 'linear'
TOPIC = 'chapter-02'
CONCEPTS = ['a matrix is a coordinate representation of a linear map, not the geometry itself', 'rank and row reduction expose how many independent constraints a system contains', 'determinants measure oriented scale and warn when inverse problems become ill conditioned', 'least squares replaces impossible exact constraints with a projection onto a column space']
VISUALS = ['grid deformation by a linear map', 'row-reduction pivots and rank', 'determinant area scale and orientation', 'least-squares residual as an orthogonal component']
CHECKS = ['matrix area scale matches the determinant', 'least-squares residual is orthogonal to the column space', 'row-reduction rank matches numpy matrix rank', 'eigenvectors reproduce their scaled directions']
SEED = 2
artifact_paths = []

In [ ]:
fig = concept_map_figure(ENTRY_TITLE, CONCEPTS, VISUALS)
concept_map_path = save_matplotlib(fig, TOPIC, "figures", "matrices-and-linear-systems-concept-route-map.png")
plt.close(fig)
artifact_paths.append(concept_map_path)
display_artifact(concept_map_path, width=820)

In [ ]:
fig = storyboard_gallery_figure(MODE, ENTRY_TITLE, VISUALS, SEED)
storyboard_gallery_path = save_matplotlib(fig, TOPIC, "figures", "storyboard-gallery.png")
plt.close(fig)
artifact_paths.append(storyboard_gallery_path)
display_artifact(storyboard_gallery_path, width=820)

In [ ]:
fig = geometry_scene_figure(MODE, ENTRY_TITLE, SEED)
geometry_scene_path = save_matplotlib(fig, TOPIC, "figures", "matrices-and-linear-systems-construction-scene.png")
plt.close(fig)
artifact_paths.append(geometry_scene_path)
display_artifact(geometry_scene_path, width=820)

## Applied Lab

Use the code cells as a starting point, then replace the supplied sample data with a case from your own graphics or geometry pipeline. For this chapter, vary one primitive, one tolerance, and one coordinate frame. Record which visual changes are geometric and which are artifacts of representation. A good lab notebook for Matrices and Linear Systems should include the input data, the rendered artifact path, the numeric residual, and a one-paragraph explanation of what would count as a failure in production code.

Suggested extension: build a second example that is nearly degenerate. Move one point close to a boundary, make two axes almost parallel, set a polynomial root almost double, or shrink a determinant toward zero. Then compare the visual artifact with the numeric check. The point of the lab is not to memorize a formula; it is to practice recognizing when a geometric answer is trustworthy, underdetermined, unstable, or dependent on a convention.

## Sanity Checklist

- matrix area scale matches the determinant
- least-squares residual is orthogonal to the column space
- row-reduction rank matches numpy matrix rank
- eigenvectors reproduce their scaled directions

The final code cell writes `final-sanity.json` into the chapter artifact subtree. The JSON is intentionally small: it records artifact sizes and a few chapter-specific numeric values so that later audits can distinguish a real teaching artifact from a blank or decorative image.

In [ ]:
fig = numerical_experiment_figure(MODE, ENTRY_TITLE, SEED)
numeric_diagnostic_path = save_matplotlib(fig, TOPIC, "figures", "numeric-diagnostic.png")
plt.close(fig)
artifact_paths.append(numeric_diagnostic_path)
display_artifact(numeric_diagnostic_path, width=820)

In [ ]:
check_values = compute_check_values(MODE)
assert check_values["max_error"] <= check_values["tolerance"], check_values
check_values

In [ ]:
require_nonempty(artifact_paths, min_bytes=1500)
final_sanity = {
    "topic": TOPIC,
    "title": ENTRY_TITLE,
    "mode": MODE,
    "artifacts": artifact_report(artifact_paths, root=BOOK_ROOT),
    "check_values": check_values,
    "checks": CHECKS,
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert sanity_path.exists() and sanity_path.stat().st_size > 200
final_sanity

## Takeaways

- Matrices and Linear Systems is a set of geometric modeling choices, not merely a list of formulas.
- The durable learning object is the combination of prose, figure, executable construction, and residual.
- Book-local artifacts make the notebook reproducible and reviewable without embedding large outputs directly in the notebook.
- A useful graphics-geometry implementation reports enough state to debug degeneracy, tolerance, and convention errors.